<div dir="rtl">

# مقایسه RNN، GRU و LSTM روی MNIST به‌صورت دنباله پیکسلی

این بخش به پیاده‌سازی و مقایسه سه نوع شبکه بازگشتی روی مجموعه داده **MNIST** اختصاص دارد. هر تصویر به‌عنوان یک دنباله ۷۸۴ گامی از پیکسل‌ها در نظر گرفته می‌شود و مدل باید در پایان دنباله، رقم را تشخیص دهد. هدف اصلی، نشان دادن تفاوت توانایی نگه‌داری **وابستگی‌های بلندمدت** در سه معماری است.

---

## ۱. مشخصات مجموعه داده (Dataset Overview)

* **نمایش ورودی:** هر تصویر $28 \times 28$ به دنباله‌ای با **طول ۷۸۴** و **ویژگی ۱** تبدیل می‌شود. شکل ورودی هر batch برابر `(B, 784, 1)` است.
* **پیش‌پردازش:** تقسیم بر ۲۵۵ و سپس نرمال‌سازی با میانگین $0.1307$ و انحراف معیار $0.3081$.
* **تقسیم داده:** ۹۰٪ آموزش (۵۴,۰۰۰ نمونه) و ۱۰٪ اعتبارسنجی (۶,۰۰۰ نمونه) با stratify روی برچسب‌ها. داده تست جدا (۱۰,۰۰۰ نمونه) فقط برای ارزیابی نهایی استفاده شده است.
* **کلاس‌های خروجی:** ۱۰ کلاس (ارقام ۰ تا ۹).
* **تابع هزینه:** `nn.CrossEntropyLoss()` (نیازمند لیبل‌های `torch.long`).
* **اندازه بچ:** ۱۲۸ برای آموزش، ۲۵۶ برای اعتبارسنجی و تست.
* **تعداد اپوک:** ۱۵۰

---

## ۲. مشخصات و پیکربندی معماری مدل‌ها

همه مدل‌ها یک ساختار یکسان دارند: **۲ لایه بازگشتی با ۶۴ واحد پنهان**، و در پایان یک لایه خطی که از گام آخر دنباله ده امتیاز کلاس می‌سازد. تنها تفاوت، نوع سلول بازگشتی است.

| مشخصه | RNN | GRU | LSTM |
| :--- | :--- | :--- | :--- |
| **تعداد پارامترهای آموزش‌پذیر** | **13,258** | **38,474** | **51,082** |
| **سلول بازگشتی** | `nn.RNN` (tanh) | `nn.GRU` | `nn.LSTM` |
| **تعداد گیت‌ها** | ۰ | ۳ (reset, update, candidate) | ۴ (input, forget, output, cell) |
| **حالت‌های پنهان** | `h` | `h` | `h` و `c` |
| **بهینه‌ساز** | `AdamW` | `AdamW` | `AdamW` |
| **نرخ یادگیری ($lr$)** | $0.0001$ | $0.0001$ | $0.0001$ |
| **Weight Decay** | $0.0001$ | $0.0001$ | $0.0001$ |
| **Gradient Clipping** | `max_norm = 1.0` | `max_norm = 1.0` | `max_norm = 1.0` |

---

## ۳. نتایج ارزیابی نهایی (Test Evaluation)

ارزیابی روی داده تست با بهترین checkpoint هر مدل (بر اساس loss اعتبارسنجی) انجام شده است.

| مدل | تعداد پارامتر | بهترین val acc | تست CELoss | دقت داده تست (Test Accuracy) |
| :--- | :---: | :---: | :---: | :---: |
| **RNN** | 13,258 | ~۰.۴۹ | ۱.۳۲۸۰ | **48.93%** |
| **LSTM** | 51,082 | ~۰.۹۲۸ | ۰.۲۵۱۸ | **92.74%** |
| **GRU** | 38,474 | ~۰.۹۴۰ | ۰.۱۹۰۶ | **94.18%** |

---

## ۴. تحلیل نتایج و گام بعدی

1. **شکست RNN ساده:** RNN حدود ۱۰۰ epoch روی حدس تصادفی ($\approx 10\%$) باقی ماند و بعد از آن به حدود ۴۹٪ رسید. دلیل اصلی، **گرادیان محوشونده** در دنباله ۷۸۴ گامی است: اطلاعات پیکسل‌های ابتدایی تصویر قبل از رسیدن به گام آخر تقریباً از بین می‌روند.
2. **برتری GRU و LSTM:** هر دو با گیت‌ها می‌توانند اطلاعات را برای مدت طولانی‌تری نگه دارند و دقتی بالای ۹۲٪ دارند. GRU با حدود ۲۵٪ پارامتر کمتر نسبت به LSTM، دقت بهتری (**۹۴.۱۸٪** در برابر **۹۲.۷۴٪**) گرفت.
3. **همگرایی ناکامل:** بهترین checkpoint های LSTM و GRU در epoch های آخر بودند، یعنی هنوز در حال بهبود بودند. اختلاف واقعی بین آن‌ها ممکن است با epoch بیشتر تغییر کند.
4. **محدودیت‌های گزارش:** هر مدل فقط با یک seed اجرا شده است و نتایج میانگین چند اجرا نیستند. برای مقایسه دقیق‌تر، اجرای چند seed توصیه می‌شود.
5. **انتقال به فاز بعد:** RNN و انواع آن برای داده‌های ترتیبی طولانی به‌تدریج جای خود را به **ترنسفورمر** داده‌اند، چون مکانیزم attention اجازه می‌دهد هر گام مستقیماً به هر گام دیگر دسترسی داشته باشد، بدون اینکه اطلاعات از زنجیره بازگشتی عبور کند. در مرحله بعد، همین داده MNIST با مدل **Vision Transformer (ViT)** بررسی خواهد شد.

</div>

In [1]:

import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import DataLoader , TensorDataset
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
df_train = pd.read_csv("../data/mnist_train.csv")
df_test = pd.read_csv("../data/mnist_test.csv")


In [4]:
torch.backends.cudnn.benchmark = True

In [5]:
df_train.head()

,label,1x1,1x2,1x3,1x4,1x5,1x6,1x7,1x8,1x9,...,28x19,28x20,28x21,28x22,28x23,28x24,28x25,28x26,28x27,28x28
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
y_train = df_train.iloc[:,0]
X_train = df_train.iloc[:,1:]


In [7]:
y_train.head()

0    5
1    0
2    4
3    1
4    9
Name: label, dtype: int64

In [8]:
X_train.head()

,1x1,1x2,1x3,1x4,1x5,1x6,1x7,1x8,1x9,1x10,...,28x19,28x20,28x21,28x22,28x23,28x24,28x25,28x26,28x27,28x28
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
X_train = X_train.values.astype("float32") / 255.0
y_train = y_train.values.astype("int64")

In [10]:
X_train , X_val , y_train , y_val = train_test_split(X_train, y_train,test_size=.1 ,shuffle= True, stratify= y_train , random_state= 43)

In [11]:
X_train = (X_train - 0.1307) / 0.3081
X_val = (X_val - 0.1307) / 0.3081

In [12]:
X_train = X_train.reshape(-1,784,1)
X_val = X_val.reshape(-1,784,1)

In [13]:
train_set = TensorDataset(torch.from_numpy(X_train),torch.from_numpy(y_train))
val_set = TensorDataset(torch.from_numpy(X_val),torch.from_numpy(y_val))

In [14]:
train_loader = DataLoader(
    train_set,
    batch_size= 128,
    shuffle=True,
    num_workers=0,
    pin_memory= True
)
val_loader = DataLoader(
    val_set,
    batch_size= 256,
    shuffle=False,
    num_workers= 0,
    pin_memory= True
)


In [15]:
class VanillaRnn(nn.Module):
    def __init__(self,input_size ,hidden_size,num_layers):
        super().__init__()
        self.rnn = nn.RNN(
            input_size= input_size,
            hidden_size = hidden_size,
            num_layers= num_layers,
            batch_first= True
        )

        self.fc = nn.Linear(hidden_size , 10)

    def forward(self,x):
        out , _ = self.rnn(x)
        last_step = out[:, -1, :]
        predict = self.fc(last_step)
        return predict

In [16]:
class Lstm(nn.Module):
    def __init__(self,input_size ,hidden_size,num_layers):
        super().__init__()
        self.rnn = nn.LSTM(
            input_size= input_size,
            hidden_size = hidden_size,
            num_layers= num_layers,
            batch_first= True
        )

        self.fc = nn.Linear(hidden_size , 10)

    def forward(self,x):
        out , _ = self.rnn(x)
        last_step = out[:, -1, :]
        predict = self.fc(last_step)
        return predict

In [17]:
class Gru(nn.Module):
    def __init__(self,input_size ,hidden_size,num_layers):
        super().__init__()
        self.rnn = nn.GRU(
            input_size= input_size,
            hidden_size = hidden_size,
            num_layers= num_layers,
            batch_first= True
        )

        self.fc = nn.Linear(hidden_size , 10)

    def forward(self,x):
        out , _ = self.rnn(x)
        last_step = out[:, -1, :]
        predict = self.fc(last_step)
        return predict

In [18]:
model_rnn = VanillaRnn(1,64,2).to(device)
model_lstm = Lstm(1,64,2).to(device)
model_gru = Gru(1,64,2).to(device)

In [19]:
criterion = nn.CrossEntropyLoss()
optimizer_rnn = torch.optim.AdamW(model_rnn.parameters(),lr = 0.0001, weight_decay=1e-4)
optimizer_lstm = torch.optim.AdamW(model_lstm.parameters(),lr = 0.0001, weight_decay=1e-4)


In [20]:
optimizer_gru = torch.optim.AdamW(model_gru.parameters(),lr = 0.0001, weight_decay=1e-4)

In [21]:
class Trainer:
    def __init__(self, model, criterion, optimizer, device, save_path="best_rnn.pt"):
        self.model = model.to(device)
        self.criterion = criterion
        self.optimizer = optimizer
        self.device = device
        self.save_path = save_path
        self.best_val_loss = float("inf")

    def _run_epoch(self, loader, train):
        self.model.train(train)  # train=True -> train mode, False -> eval mode
        total_loss, correct, total = 0.0, 0, 0

        with torch.set_grad_enabled(train):
            for x, y in loader:
                x, y = x.to(self.device), y.to(self.device)

                out = self.model(x)
                loss = self.criterion(out, y)

                if train:
                    self.optimizer.zero_grad()
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

                    self.optimizer.step()

                total_loss += loss.item() * x.size(0)
                correct += (out.argmax(dim=1) == y).sum().item()
                total += x.size(0)

        return total_loss / total, correct / total

    def fit(self, train_loader, val_loader, epochs):
        for epoch in range(epochs):
            train_loss, train_acc = self._run_epoch(train_loader, train=True)
            val_loss, val_acc = self._run_epoch(val_loader, train=False)

            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                torch.save(self.model.state_dict(), self.save_path)
                print(f"save!!! Epoch {epoch:02d}/{epochs} | val CELoss: {val_loss:.4f} | val acc: {val_acc:.4f}")

            if epoch % 10 == 0 or epoch == epochs - 1:
                print(f"Epoch {epoch:02d}/{epochs} | train CELoss: {train_loss:.4f} acc: {train_acc:.4f} "
                      f"| val CELoss: {val_loss:.4f} acc: {val_acc:.4f}")
        return self

In [22]:
trainer = Trainer(
    model=model_rnn,
    criterion=nn.CrossEntropyLoss(),
    optimizer=optimizer_rnn,
    device=device,
    save_path="best_rnn.pt",
)
trainer.fit(train_loader, val_loader, epochs=150)

save!!! Epoch 00/150 | val CELoss: 2.3012 | val acc: 0.1123
Epoch 00/150 | train CELoss: 2.3021 acc: 0.1108 | val CELoss: 2.3012 acc: 0.1123
Epoch 10/150 | train CELoss: 2.3014 acc: 0.1124 | val CELoss: 2.3013 acc: 0.1123
save!!! Epoch 16/150 | val CELoss: 2.3012 | val acc: 0.1123
save!!! Epoch 17/150 | val CELoss: 2.3012 | val acc: 0.1123
save!!! Epoch 19/150 | val CELoss: 1.9509 | val acc: 0.2790
Epoch 20/150 | train CELoss: 1.9411 acc: 0.2774 | val CELoss: 1.9754 acc: 0.2673
save!!! Epoch 21/150 | val CELoss: 1.9007 | val acc: 0.2963
save!!! Epoch 23/150 | val CELoss: 1.8926 | val acc: 0.2960
save!!! Epoch 24/150 | val CELoss: 1.8872 | val acc: 0.2942
save!!! Epoch 25/150 | val CELoss: 1.8829 | val acc: 0.3015
save!!! Epoch 30/150 | val CELoss: 1.8778 | val acc: 0.2998
Epoch 30/150 | train CELoss: 1.8906 acc: 0.2916 | val CELoss: 1.8778 acc: 0.2998
save!!! Epoch 31/150 | val CELoss: 1.8668 | val acc: 0.2978
save!!! Epoch 32/150 | val CELoss: 1.8572 | val acc: 0.3037
save!!! Epoch 34

In [23]:
trainer = Trainer(
    model=model_lstm,
    criterion=nn.CrossEntropyLoss(),
    optimizer=optimizer_lstm,
    device=device,
    save_path="best_lstm.pt",
)
trainer.fit(train_loader, val_loader, epochs=150)

save!!! Epoch 00/150 | val CELoss: 2.3013 | val acc: 0.1123
Epoch 00/150 | train CELoss: 2.3023 acc: 0.1107 | val CELoss: 2.3013 acc: 0.1123
save!!! Epoch 01/150 | val CELoss: 2.3012 | val acc: 0.1123
save!!! Epoch 03/150 | val CELoss: 2.3012 | val acc: 0.1123
save!!! Epoch 04/150 | val CELoss: 2.3012 | val acc: 0.1123
save!!! Epoch 05/150 | val CELoss: 2.1116 | val acc: 0.2090
save!!! Epoch 06/150 | val CELoss: 1.8631 | val acc: 0.2557
save!!! Epoch 07/150 | val CELoss: 1.7168 | val acc: 0.3252
save!!! Epoch 08/150 | val CELoss: 1.6667 | val acc: 0.3772
Epoch 10/150 | train CELoss: 1.8772 acc: 0.3331 | val CELoss: 1.9289 acc: 0.3008
save!!! Epoch 12/150 | val CELoss: 1.5601 | val acc: 0.4725
save!!! Epoch 13/150 | val CELoss: 1.5241 | val acc: 0.4658
save!!! Epoch 20/150 | val CELoss: 1.4949 | val acc: 0.4427
Epoch 20/150 | train CELoss: 1.5292 acc: 0.4443 | val CELoss: 1.4949 acc: 0.4427
save!!! Epoch 21/150 | val CELoss: 1.4412 | val acc: 0.4643
save!!! Epoch 23/150 | val CELoss: 1.

In [24]:
trainer = Trainer(
    model=model_gru,
    criterion=nn.CrossEntropyLoss(),
    optimizer=optimizer_gru,
    device=device,
    save_path="best_gru.pt",
)
trainer.fit(train_loader, val_loader, epochs=150)

save!!! Epoch 00/150 | val CELoss: 2.3012 | val acc: 0.1123
Epoch 00/150 | train CELoss: 2.3023 acc: 0.1103 | val CELoss: 2.3012 acc: 0.1123
save!!! Epoch 01/150 | val CELoss: 2.3011 | val acc: 0.1123
save!!! Epoch 02/150 | val CELoss: 2.0434 | val acc: 0.2125
save!!! Epoch 03/150 | val CELoss: 1.7287 | val acc: 0.3440
save!!! Epoch 04/150 | val CELoss: 1.7256 | val acc: 0.3522
save!!! Epoch 05/150 | val CELoss: 1.6972 | val acc: 0.3513
save!!! Epoch 06/150 | val CELoss: 1.6902 | val acc: 0.3593
save!!! Epoch 07/150 | val CELoss: 1.6728 | val acc: 0.3882
save!!! Epoch 08/150 | val CELoss: 1.6604 | val acc: 0.3893
save!!! Epoch 09/150 | val CELoss: 1.6324 | val acc: 0.4182
save!!! Epoch 10/150 | val CELoss: 1.6152 | val acc: 0.4293
Epoch 10/150 | train CELoss: 1.6396 acc: 0.4187 | val CELoss: 1.6152 acc: 0.4293
save!!! Epoch 11/150 | val CELoss: 1.6023 | val acc: 0.4260
save!!! Epoch 12/150 | val CELoss: 1.5976 | val acc: 0.4237
save!!! Epoch 13/150 | val CELoss: 1.5910 | val acc: 0.415

In [25]:
X_test = df_test.iloc[:, 1:].values.astype("float32") / 255.0
y_test = df_test.iloc[:, 0].values.astype("int64")

X_test = (X_test - 0.1307) / 0.3081
X_test = X_test.reshape(-1, 784, 1)

test_set = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))
test_loader = DataLoader(test_set, batch_size=256, shuffle=False,
                         num_workers=0, pin_memory=True)

In [26]:
def evaluate_checkpoint(model, ckpt_path, loader, criterion, device):
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            total_loss += criterion(out, y).item() * x.size(0)
            correct += (out.argmax(dim=1) == y).sum().item()
            total += x.size(0)

    return total_loss / total, correct / total


criterion = nn.CrossEntropyLoss()
results = {}

for name, model, ckpt in [
    ("RNN",  model_rnn,  "best_rnn.pt"),
    ("LSTM", model_lstm, "best_lstm.pt"),
    ("GRU",  model_gru,  "best_gru.pt"),
]:
    loss, acc = evaluate_checkpoint(model, ckpt, test_loader, criterion, device)
    results[name] = (loss, acc)
    print(f"{name:5s} | test CELoss: {loss:.4f} | test acc: {acc:.4f}")

RNN   | test CELoss: 1.3280 | test acc: 0.4893
LSTM  | test CELoss: 0.2518 | test acc: 0.9274
GRU   | test CELoss: 0.1906 | test acc: 0.9418
